In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import h3
import plotly.express as px 
import geopy.distance

os.environ['HAVEN_DATABASE'] = 'haven'
os.environ['AWS_PROFILE'] = 'admin'

from mirrorverse.utils import read_data_w_cache
from mirrorverse.plotting import plot_h3_slider

In [ ]:
from shapely.geometry import Polygon, Point

poly = Polygon(
    [
        (-166, 54.4),
        (-160, 56),
        (-158, 57.2),
        (-153, 62),
        (-149, 62),
        (-146, 62),
        (-140, 60),
        (-136, 58.4),
        (-133, 57.5),
        (-132, 56.0),
        (-131, 55),
        (-125, 50.3),
        (-170, 52.5),
        (-166, 54.4),
    ]
)

In [ ]:
sql = '''
select
    *
from 
    mean_elevation_by_h3
where 
    h3_resolution = 4
    and elevation > -600
'''
allowed = read_data_w_cache(sql)
allowed.head()

sql = '''
select 
    origin_h3_index,
    next_h3_index,
    time,
    probability,
    stay_put
from 
    movement_model_full_inference_10_1_10
where 
    time = TIMESTAMP '2022-03-15 12:00:00'
'''
data = read_data_w_cache(sql)
print(data.shape)
data['lat'] = data['origin_h3_index'].apply(lambda h: h3.h3_to_geo(h)[0])
data['lon'] = data['origin_h3_index'].apply(lambda h: h3.h3_to_geo(h)[1])

data['inside_polygon'] = data.apply(lambda row: poly.contains(Point(row['lon'], row['lat'])), axis=1)
data = data[data['inside_polygon'] & (data['lon'] < -145)].merge(
    allowed[['h3_index']].rename(columns={'h3_index': 'origin_h3_index'})
)

#data = data[
#    (data['lat'] > 47) & (data['lat'] < 65)
#    & (data['lon'] < -120) & (data['lat'] > -176)
#]
origins = sorted(data['origin_h3_index'].unique())
data = data[data['next_h3_index'].isin(origins)]
data['probability'] = data['probability'].fillna(0) + 0.00001
data['total_probability'] = data.groupby('origin_h3_index')['probability'].transform('sum')
data['probability'] = data['probability'] / data['total_probability']
print(data.shape)
data.head()

In [ ]:
data[data['origin_h3_index'] == origins[1]]

In [ ]:
indices = {
    h3_index: i 
    for i, h3_index in enumerate(origins)
}
len(indices)

In [ ]:
num_options = data.groupby('origin_h3_index').size().to_dict()

In [ ]:
M = np.zeros((len(indices), len(indices)))
N = np.zeros((len(indices), len(indices)))

for _, row in tqdm(data.iterrows()):
    num_neighors = num_options[row['origin_h3_index']]
    i = indices[row['next_h3_index']]
    j = indices[row['origin_h3_index']]
    M[i, j] = row['probability']
    M[i, j] = row['probability']
    N[i, j] = 1 / num_neighors

In [ ]:
MX = np.linalg.matrix_power(M, 1)
NX = np.linalg.matrix_power(N, 1)

In [ ]:
import scipy

def negative_entropy(p):
    return (p[p > 0] * np.log(p[p > 0])).sum()

delta = 0.025
p = np.ones(MX.shape[0])
p = p / p.sum()

bounds = [
    (0, 1) for _ in p
]

constraints = [
    scipy.optimize.LinearConstraint(
        np.ones(p.shape[0]), lb=1, ub=1
    ),
    scipy.optimize.LinearConstraint(
        MX - np.identity(p.shape[0]) * (1 + delta), lb=-float("inf"), ub=0
    ),
    scipy.optimize.LinearConstraint(
        MX - np.identity(p.shape[0]) * (1 - delta), lb=0, ub=float("inf")
    )
]

result = scipy.optimize.minimize(
    negative_entropy, p, 
    bounds=bounds, 
    constraints=constraints,
    options={'maxiter': 100}
)
result

In [ ]:
np.max(np.abs((result.x - np.matmul(MX, result.x)) / result.x))

In [ ]:
reverse_index = {
    i: h3_index 
    for h3_index, i in indices.items()
}

rows = [
    {
        'h3_index': reverse_index[i],
        'probability': p / result.x.max(),
        'version': 1
    } for i, p in enumerate(result.x)
]
chaos = pd.DataFrame(rows)
print(chaos.shape)
chaos.head()

In [ ]:
px.histogram(chaos['probability'])

In [ ]:
plot_h3_slider(
    chaos, 'probability', 'h3_index', 'version', zmin=0, zmax=chaos['probability'].quantile(0.95)
)

In [ ]:
def solve_it(rho, delta, MX):
    total = np.sum(rho)
    num_changes = float('inf')
    iterations = 0
    while num_changes > 0:
        rhot = np.matmul(MX, rho)
        perc_dif = np.abs(rhot - rho) / rho
        rho[perc_dif > delta] = rhot[perc_dif > delta]
        rho = rho / np.sum(rho) * total

        num_changes = np.sum(perc_dif > delta)
        iterations += 1
    return rho, iterations

In [ ]:
rhos = []
iters = []
for i in tqdm(range(25)):
    rho = np.random.random(len(indices)) 
    rho = rho / np.sum(rho)
    rho, iterations = solve_it(rho, 0.025, MX)
    iters.append(iterations)
    rhos.append(rho)
rho = np.array(rhos).mean(axis=0)

In [ ]:
reverse_index = {
    i: h3_index 
    for h3_index, i in indices.items()
}

In [ ]:
rows = []
for i, val in enumerate(rho):
    rows.append({
        'h3_index': reverse_index[i],
        'val': val,
        'version': 1
    })
rho_df = pd.DataFrame(rows)
rho_df.head()

In [ ]:
np.abs(np.matmul(MX, rho) - rho) / rho

In [ ]:
plot_h3_slider(
    rho_df, 'val', 'h3_index', 'version', zmin=0, zmax=rho_df['val'].quantile(0.95)
)

In [ ]:
from shapely.geometry import Polygon, Point

poly = Polygon(
    [
        (-166, 54.4),
        (-160, 56),
        (-158, 57.2),
        (-153, 62),
        (-149, 62),
        (-146, 62),
        (-140, 60),
        (-136, 58.4),
        (-133, 57.5),
        (-132, 56.0),
        (-131, 55),
        (-125, 50.3),
        (-170, 52.5),
        (-166, 54.4),
    ]
)
rho_df['lat'] = rho_df['h3_index'].apply(lambda h: h3.h3_to_geo(h)[0])
rho_df['lon'] = rho_df['h3_index'].apply(lambda h: h3.h3_to_geo(h)[1])
rho_df['inside_polygon'] = rho_df.apply(lambda row: poly.contains(Point(row['lon'], row['lat'])), axis=1)
rho_df.head()

In [ ]:
sql = '''
select
    *
from 
    mean_elevation_by_h3
where 
    h3_resolution = 4
    and elevation > -600
'''
allowed = read_data_w_cache(sql)
allowed.head()

In [ ]:
df = rho_df[rho_df['inside_polygon'] & (rho_df['lon'] < -145)].merge(allowed[['h3_index']])
df['log_val'] = np.log(df['val'] / df['val'].min()) / np.log(2)
plot_h3_slider(df, 'log_val', 'h3_index', 'version', zmin=df['log_val'].quantile(0.05), zmax=df['log_val'].quantile(0.95))

In [ ]:
np.log(4)/np.log(2)

In [ ]:
MX = np.linalg.matrix_power(M, 1)
NX = np.linalg.matrix_power(N, 1)

In [ ]:
rows = []
for h3_index, i in indices.items():
    P = MX[:, i]
    Q = NX[:, i]
    P = P[Q != 0]
    Q = Q[Q != 0]
    divergence = sum(P * np.log(P/Q)) 
    rows.append({
        'h3_index': h3_index,
        'divergence': divergence,
    })
divergence = pd.DataFrame(rows)
print(divergence.shape)
divergence.head()

In [ ]:
divergence['version'] = 1
plot_h3_slider(divergence, 'divergence', 'h3_index', 'version', zmin=0, zmax=divergence['divergence'].quantile(0.9)).show()

In [ ]:
reverse_index = {
    i: h3_index 
    for h3_index, i in indices.items()
}

In [ ]:
rows = []
for h3_index, i in tqdm(indices.items()):
    P = MX[:, i]
    Q = NX[:, i]
    for j, (p, q) in enumerate(zip(P, Q)):
        if q != 0:
            rows.append({
                'origin_h3_index': h3_index,
                'h3_index': reverse_index[j],
                'shift': p-q#p * np.log(p/q)#p - q
            })
shifts = pd.DataFrame(rows)
print(shifts.shape)
shifts.head()

In [ ]:
h3_index = '840cd99ffffffff'#'8422811ffffffff'#'84228a3ffffffff'
shifts['color'] = shifts.apply(lambda r: 'red' if r['origin_h3_index'] == r['h3_index'] else 'blue', axis=1)
df = shifts[shifts['origin_h3_index'] == h3_index]
boundary = max(abs(df['shift'].min()), df['shift'].max())
plot_h3_slider(
    shifts[shifts['origin_h3_index'] == h3_index], 'shift', 'h3_index', 'origin_h3_index', line_color_col='color', bold_colors=['red'],
    colorscale='RdBu', zmin=-boundary, zmax=boundary
)

In [ ]:
rows = []
for h3_index, i in tqdm(indices.items()):
    origin = h3_index
    olat, olon = h3.h3_to_geo(origin)
    P = MX[:, i]
    Q = NX[:, i]
    for j, (p, q) in enumerate(zip(P, Q)):
        if q != 0:
            dest = reverse_index[j]
            dlat, dlon = h3.h3_to_geo(dest)

            E = geopy.distance.geodesic((olat, dlon), (olat, olon)).km 
            N = geopy.distance.geodesic((dlat, olon), (olat, olon)).km

            E = -E if dlon < olon else E 
            N = -N if dlat < olat else N

            rows.append({
                'origin_h3_index': h3_index,
                'h3_index': reverse_index[j],
                'N': N,
                'E': E,
                'mass': p
            })
mass = pd.DataFrame(rows)

mass['N_mass'] = mass['N'] * mass['mass']
mass['E_mass'] = mass['E'] * mass['mass']
mass['D_mass'] = (mass['N'] ** 2 + mass['E'] ** 2) ** 0.5 * mass['mass']

print(mass.shape)
mass.head()

In [ ]:
center = mass.groupby('origin_h3_index')[['N_mass', 'E_mass', 'D_mass', 'mass']].sum().reset_index()
center['N'] = center['N_mass'] / center['mass']
center['E'] = center['E_mass'] / center['mass']
center['D'] = center['D_mass'] / center['mass']
center['distance'] = (center['N'] ** 2 + center['E'] ** 2) ** 0.5 
center['log_distance'] = np.log(center['distance'] + 1)
center['norm_N'] = center['N'] / center['distance']
center['norm_E'] = center['E'] / center['distance']
center['version'] = 1
center.head() 

In [ ]:
px.histogram(center['D'])

In [ ]:
plot_h3_slider(center, 'distance', 'origin_h3_index', 'version', zmax=center['distance'].quantile(0.9), zmin=0)

In [ ]:
from shapely.geometry import Polygon, Point

poly = Polygon(
    [
        (-166, 54.4),
        (-160, 56),
        (-158, 57.2),
        (-153, 62),
        (-149, 62),
        (-146, 62),
        (-140, 60),
        (-136, 58.4),
        (-133, 57.5),
        (-132, 56.0),
        (-131, 55),
        (-125, 50.3),
        (-170, 52.5),
        (-166, 54.4),
    ]
)
center['lat'] = center['origin_h3_index'].apply(lambda h: h3.h3_to_geo(h)[0])
center['lon'] = center['origin_h3_index'].apply(lambda h: h3.h3_to_geo(h)[1])
center['inside_polygon'] = center.apply(lambda row: poly.contains(Point(row['lon'], row['lat'])), axis=1)
center.head()

In [ ]:
sql = '''
select
    *
from 
    mean_elevation_by_h3
where 
    h3_resolution = 4
    and elevation > -600
'''
allowed = read_data_w_cache(sql)
allowed.head()

In [ ]:
plot_h3_slider(center[center['inside_polygon'] & (center['lon'] < -145)].merge(allowed[['h3_index']].rename(columns={'h3_index': 'origin_h3_index'})), 'distance', 'origin_h3_index', 'version', zmin=0)

In [ ]:
plot_h3_slider(center[center['inside_polygon'] & (center['lon'] < -145)].merge(allowed[['h3_index']].rename(columns={'h3_index': 'origin_h3_index'})), 'D', 'origin_h3_index', 'version', zmin=0, zmax=20)

In [ ]:
angle = np.pi/2
E = np.cos(angle)
N = np.sin(angle)

center['contribution'] = (center['norm_N'] * N + center['norm_E'] * E)
df = center[center['inside_polygon'] & (center['lon'] < -145)].merge(allowed[['h3_index']].rename(columns={'h3_index': 'origin_h3_index'})).copy()
df = df[df['distance'] > 2]
plot_h3_slider(
    df, 
    'contribution', 'origin_h3_index', 'version', zmin=-1.0, zmax=1.0, colorscale='RdBu'
)

In [ ]:
angle = 0
E = np.cos(angle)
N = np.sin(angle)

center['contribution'] = (center['norm_N'] * N + center['norm_E'] * E)
df = center[center['inside_polygon'] & (center['lon'] < -145)].merge(allowed[['h3_index']].rename(columns={'h3_index': 'origin_h3_index'})).copy()
df = df[df['distance'] > 2]
plot_h3_slider(
    df, 
    'contribution', 'origin_h3_index', 'version', zmin=-1.0, zmax=1.0, colorscale='RdBu'
)

In [ ]:
info = center[['origin_h3_index', 'N', 'E', 'D']].set_index('origin_h3_index').to_dict()
S = np.zeros((center.shape[0], center.shape[0]))
A = np.zeros((center.shape[0], center.shape[0]))
K = np.zeros((center.shape[0], center.shape[0]))
for origin, i in tqdm(indices.items()):
    for dest, j in indices.items():
        if j < i: continue
        N = info['N'][origin] - info['N'][dest]
        E = info['E'][origin] - info['E'][dest]
        D = abs(info['D'][origin] - info['D'][dest])
        K[i,j] = geopy.distance.geodesic(h3.h3_to_geo(origin), h3.h3_to_geo(dest)).km
        K[j,i] = K[i,j]
        L = (N ** 2 + E ** 2) ** 0.5
        S[i, j] = D
        S[j, i] = D
        A[i, j] = L
        A[j, i] = L

In [ ]:
SN = 1 - (S / np.quantile(S, 0.98))
AN = 1 - (A / np.quantile(A, 0.98))
AN[AN < 0] = 0
SN[SN < 0] = 0
KN = K.copy()
KN[KN == 0] = KN[KN != 0].min()
KN = KN / KN.min()
KN.min()

In [ ]:
a = 0
s = 1
R = (AN * a + SN * s) / (a + s) #/ KN
R = 1 - R
R

In [ ]:
from sklearn.cluster import AgglomerativeClustering

clustering = AgglomerativeClustering(
    n_clusters=5,
    metric="precomputed",
    linkage="average",
)
clustering.fit(R)

In [ ]:
rows = []
for i, label in enumerate(clustering.labels_):
    rows.append({
        'h3_index': reverse_index[i],
        'label': label
    })
labels = pd.DataFrame(rows)
labels['version'] = 1
plot_h3_slider(labels, 'label', 'h3_index', 'version')


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=1)
X = np.array(center[['N', 'E']])
X = X - X.mean(axis=0)
pca.fit(X)
pca.explained_variance_ratio_

In [ ]:
center['C'] = 

In [ ]:
X.mean(axis=0)

In [ ]:
rows = []
for h3_index, i in tqdm(indices.items()):
    origin = h3_index
    olat, olon = h3.h3_to_geo(origin)
    P = MX[:, i]
    Q = NX[:, i]
    for j, (p, q) in enumerate(zip(P, Q)):
        if q != 0:
            dest = reverse_index[j]
            dlat, dlon = h3.h3_to_geo(dest)
            NS = dlat - olat
            EW = dlon - olon
            length = (NS ** 2 + EW ** 2) ** 0.5
            rows.append({
                'origin_h3_index': h3_index,
                'h3_index': reverse_index[j],
                'N': NS, #/ length if length != 0 else 0,
                'E': EW, #/ length if length != 0 else 0,
                'shift': p
            })
shifts = pd.DataFrame(rows)
print(shifts.shape)
shifts.head()

In [ ]:
angle = np.pi/2
E = np.cos(angle)
N = np.sin(angle)

shifts['contribution'] = (shifts['N'] * N + shifts['E'] * E) * shifts['shift']
df = shifts.groupby('origin_h3_index')[['contribution', 'shift']].sum().reset_index()
df['version'] = 1
boundary = max(abs(df['contribution'].quantile(0.1)), abs(df['contribution'].quantile(0.9)))
plot_h3_slider(df, 'contribution', 'origin_h3_index', 'version', zmin=-boundary, zmax=boundary, colorscale='RdBu')

In [ ]:
angle = 0
E = np.cos(angle)
N = np.sin(angle)

shifts['contribution'] = (shifts['N'] * N + shifts['E'] * E) * shifts['shift']
df = shifts.groupby('origin_h3_index')[['contribution', 'shift']].sum().reset_index()
df['version'] = 1
boundary = max(abs(df['contribution'].quantile(0.1)), abs(df['contribution'].quantile(0.9)))
plot_h3_slider(df, 'contribution', 'origin_h3_index', 'version', zmin=-boundary, zmax=boundary, colorscale='RdBu')

In [ ]:
rows = [
    {
        'h3_index': h3_index,
        'stickiness': MX[i, i]
    }
    for h3_index, i in indices.items()
]
df = pd.DataFrame(rows)
df['version'] = 1
plot_h3_slider(
    df, 'stickiness', 'h3_index', 'version', zmin=df['stickiness'].quantile(0.5), zmax=df['stickiness'].quantile(0.9)
)

In [ ]:
rows = []
for h3_index, i in tqdm(indices.items()):
    origin = h3_index
    olat, olon = h3.h3_to_geo(origin)
    P = MX[:, i]
    Q = NX[:, i]
    for j, (p, q) in enumerate(zip(P, Q)):
        if q != 0:
            dest = reverse_index[j]
            dlat, dlon = h3.h3_to_geo(dest)
            NS = dlat - olat
            EW = dlon - olon
            length = (NS ** 2 + EW ** 2) ** 0.5
            rows.append({
                'origin_h3_index': h3_index,
                'h3_index': reverse_index[j],
                'N': NS, #/ length if length != 0 else 0,
                'E': EW, #/ length if length != 0 else 0,
                'p': p,
                'q': q
            })
shifts = pd.DataFrame(rows)
print(shifts.shape)
shifts.head()

In [ ]:
shifts['q_N'] = shifts['N'] * shifts['q'] 
shifts['q_E'] = shifts['E'] * shifts['q'] 
shifts['p_N'] = shifts['N'] * shifts['p'] 
shifts['p_E'] = shifts['E'] * shifts['p'] 
df = shifts.groupby('origin_h3_index')[['q_N', 'q_E', 'q', 'p_N', 'p_E', 'p']].sum().reset_index()
df['q_N'] = df['q_N'] / df['q']
df['q_E'] = df['q_E'] / df['q']
df['q_anisotropy'] = (df['q_N'] ** 2 + df['q_E'] ** 2) ** 0.5
df['p_N'] = df['p_N'] / df['p']
df['p_E'] = df['p_E'] / df['p']
df['p_anisotropy'] = (df['p_N'] ** 2 + df['p_E'] ** 2) ** 0.5


In [ ]:
df['version'] = 1
df['anisotropy'] = df['p_anisotropy'] - df['q_anisotropy']
plot_h3_slider(
    df, 'anisotropy', 'origin_h3_index', 'version', zmin=df['anisotropy'].quantile(0.1), zmax=df['anisotropy'].quantile(0.9)
)

In [ ]:
angle = np.pi/2
E = np.cos(angle)
N = np.sin(angle)
df['value'] = df['p_E'] * E + df['p_N'] * N
boundary = max(abs(df['value'].quantile(0.1)), abs(df['value'].quantile(0.9)))
plot_h3_slider(
    df, 'value', 'origin_h3_index', 'version', zmin=-boundary, zmax=boundary, colorscale='RdBu'
)

In [ ]:
angle = 0
E = np.cos(angle)
N = np.sin(angle)
df['value'] = df['p_E'] * E + df['p_N'] * N
boundary = max(abs(df['value'].quantile(0.1)), abs(df['value'].quantile(0.9)))
plot_h3_slider(
    df, 'value', 'origin_h3_index', 'version', zmin=-boundary, zmax=boundary, colorscale='RdBu'
)